In [1]:
from transformers import AutoImageProcessor, AutoModelForDepthEstimation
import torch
import numpy as np
from PIL import Image
import requests
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from datetime import datetime
import logging
import supervision as sv
import pathlib
from tqdm import tqdm
import pandas as pd
from collections import defaultdict

/Users/S100AHBD/Desktop/CityVision/cityvision/test_cityvision/lib/python3.11/site-packages/torchvision/datapoints/__init__.py:12: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may still change according to user feedback. Please submit any feedback you may have in this issue: https://github.com/pytorch/vision/issues/6753, and you can also check out https://github.com/pytorch/vision/issues/7319 to learn more about the APIs that we suspect might involve future changes. You can silence this warning by calling torchvision.disable_beta_transforms_warning().
  warnings.warn(_BETA_TRANSFORMS_WARNING)
/Users/S100AHBD/Desktop/CityVision/cityvision/test_cityvision/lib/python3.11/site-packages/torchvision/transforms/v2/__init__.py:54: UserWarning: The torchvision.datapoints and torchvision.transforms.v2 namespaces are still Beta. While we do not expect major breaking changes, some APIs may s

In [2]:
# Initialize the camera matrix and distortion coefficients, speed zone
fx = 480.35583204
fy = 437.11894211
cx = 354.90375117
cy = 249.48026187

dis_vec = np.array(
    [-4.09277234e-01, 1.89529196e-01, -2.03433425e-04, -2.01818939e-03, -5.22683299e-02]
)

# Create camera matrix
camera_matrix = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])

speed_zone = [[0, 480], [720, 480], [720, 0], [0, 0]]

In [3]:
# function for undistorting the frame and crop it
def undistort(frame):
    h, w = frame.shape[:2]
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dis_vec, (w, h), 1, (w, h)
    )
    dst = cv2.undistort(frame, camera_matrix, dis_vec, None, newcameramtx)
    x, y, w, h = roi
    dst = dst[y : y + h, x : x + w]

    dst = Image.fromarray(dst)

    return dst

In [4]:
def undistort_pt(pt_matrix, frame):
    h, w = frame.shape[:2]
    print(f"Frame shape: {frame.shape}")
    print(f"Point matrix shape: {pt_matrix.shape}")
    newcameramtx, roi = cv2.getOptimalNewCameraMatrix(
        camera_matrix, dis_vec, (w, h), 1, (w, h)
    )
    undst_pt = cv2.undistortPoints(
        pt_matrix, camera_matrix, dis_vec, None, P=newcameramtx
    )
    undst_pt = undst_pt.squeeze()
    x, y, w, h = roi
    undst_pt[0] -= x
    undst_pt[1] -= y
    return undst_pt

In [5]:
# function to get the depth map from the frist frame


def get_depth_map(frame):

    CHECKPOINT = "depth-anything/Depth-Anything-V2-Metric-Outdoor-Large-hf"

    image_processor = AutoImageProcessor.from_pretrained(CHECKPOINT)
    model = AutoModelForDepthEstimation.from_pretrained(CHECKPOINT)

    inputs = image_processor(images=frame, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    prediction = (
        torch.nn.functional.interpolate(
            predicted_depth.unsqueeze(1),
            size=frame.size[::-1],
            mode="bicubic",
            align_corners=False,
        )
        .squeeze(0)
        .squeeze(0)
        .cpu()
        .numpy()
    )

    return prediction

In [6]:
# calculate distance between two points using depth map and camera matrix


def calculate_distance(depth_map, camera_matrix, x1, y1, x2, y2):
    # Get the depth values at the two points

    if y1 < 0 or y1 >= depth_map.shape[0] or x1 < 0 or x1 >= depth_map.shape[1]:
        return "N/A"

    if y2 < 0 or y2 >= depth_map.shape[0] or x2 < 0 or x2 >= depth_map.shape[1]:
        return "N/A"

    z1 = depth_map[y1, x1]
    z2 = depth_map[y2, x2]

    # Calculate the 3D coordinates of the two points
    p1 = np.array(
        [
            (x1 - camera_matrix[0, 2]) * z1 / camera_matrix[0, 0],
            (y1 - camera_matrix[1, 2]) * z1 / camera_matrix[1, 1],
            z1,
        ]
    )
    p2 = np.array(
        [
            (x2 - camera_matrix[0, 2]) * z2 / camera_matrix[0, 0],
            (y2 - camera_matrix[1, 2]) * z2 / camera_matrix[1, 1],
            z2,
        ]
    )

    # Calculate the distance between the two points
    distance = np.linalg.norm(p2 - p1) / 1.5

    return distance

In [7]:
# getting depth map from the first frame

image = Image.open("../assets/test3.jpg")
frame = np.array(image)
frame = undistort(frame)
depth_map = get_depth_map(frame)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


# Speed calculation for cars in video

In [8]:
# initiate yolo11 for car detection and tracking
model = YOLO("yolo11l.pt")
confidence_threshold = 0.2
tracker_config = "../configs/bytetrack.yml"

In [ ]:
def get_count(
    frame,
    start_time,
    config,
    speeds,
    model,
    speed_zone,
    crossed_objects,
    depth_map,
    track_history,
    count,
) -> dict:

    results, boxes, class_ids, class_names, annotated_frame = get_result(
        frame, config, model
    )

    if results[0].boxes.id is not None:
        track_ids = results[0].boxes.id.cpu().int().tolist()

        # Plot the tracks and count objects crossing the line
        for box, track_id, cls in zip(boxes, track_ids, class_names):
            x, y, w, h = box
            pt = (int(x.numpy()), int(y.numpy()))
            cls = cls
            track = track_history[track_id]
            track.append((float(x), float(y)))  # x, y center point

            if len(track) > 60:  # retain 60 tracks for 60 frames
                track.pop(0)
            # Check if the object crosses the line
            if track_id not in crossed_objects["EB"]:
                time_seen = datetime.fromtimestamp(
                    int(count / 30) + start_time.timestamp()
                )
                crossed_objects["EB"][track_id] = [
                    time_seen.strftime("%Y-%m-%d %H:%M:%S"),
                    cls,
                ]

            # calculate the speed if object in speed zone based on track history
            if len(track) > 10:

                undst_pt_1 = undistort_pt(np.array(track[-1]), frame)
                print(f" track[-1]: {track[-1]}")
                print(f" undst_pt_1: {undst_pt_1}")
                undst_pt_2 = undistort_pt(np.array(track[-10]), frame)
                x1, y1 = undst_pt_1[0], undst_pt_1[1]
                x2, y2 = undst_pt_2[0], undst_pt_2[1]
                # x1, y1 = track[-1]
                # x2, y2 = track[0]

                distance = calculate_distance(
                    depth_map,
                    camera_matrix,
                    int(x2),
                    int(y2),
                    int(x1),
                    int(y1),
                )
                if distance != "N/A":
                    time = 10 / 30
                    speed = distance / time
                    speeds[track_id].append(speed * 3.6)
                    # exponentially smooth the speed for last 30 frames
                    if len(speeds[track_id]) > 30:
                        speeds[track_id].pop(0)
                # show speed on the frame
                if track_id in speeds:
                    cv2.putText(
                        annotated_frame,
                        f"{np.mean(speeds[track_id]):.2f} km/h",
                        (int(x - w / 2), int(y - h / 2)),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1,
                        (0, 255, 0),
                        2,
                    )

                # Annotate the object as it crosses the line
                cv2.rectangle(
                    annotated_frame,
                    (int(x - w / 2), int(y - h / 2)),
                    (int(x + w / 2), int(y + h / 2)),
                    (0, 255, 0),
                    2,
                )

            # Annotate center of the object
            cv2.circle(annotated_frame, pt, 5, (0, 255, 0), -1)

    return annotated_frame


def get_result(frame, config, model) -> dict:

    class_ids = list(config["class_ids"].keys())
    results = model.track(
        frame,
        classes=class_ids,
        persist=True,
        save=False,
        tracker=config["tracker_config"],
        verbose=False,
        conf=config["conf"],
        iou=config["iou"],
        agnostic_nms=False,
    )

    # Get the boxes and track IDs
    boxes = results[0].boxes.xywh.cpu()

    class_ids = results[0].boxes.cls.cpu().int().tolist()

    class_names = [config["class_ids"][i] for i in class_ids]

    # Visualize the results on the frame
    annotated_frame = results[0].plot()

    return (results, boxes, class_ids, class_names, annotated_frame)


def resample_data(df_first: pd.DataFrame, interval: str = "1min") -> pd.DataFrame:

    df = df_first.copy()
    # Reset the index to make the track_id a column
    df.reset_index(inplace=True)
    # Set the index to the timestamp
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df.set_index("timestamp", inplace=True)
    df.rename(columns={"index": "Track ID"}, inplace=True)
    # Resample the data by intervals
    resampled_df = df.resample(interval).count()

    # Reset the index to make the time intervals a column
    resampled_df = resampled_df.reset_index()

    # Rename the time interval column to 'timestep'
    resampled_df.rename(columns={"index": "timestep"}, inplace=True)

    return resampled_df.copy()


def generate_report(uuid, crossed_objects) -> pd.DataFrame:

    print(crossed_objects)

    df_1 = pd.DataFrame.from_dict(
        crossed_objects["EB"], orient="index", columns=["timestamp", "Class"]
    )

    df_1 = resample_data(df_1)

    df_1["Direction"] = "EB"

    resampled_df = df_1.copy()
    resampled_df["uuid"] = uuid

    print("df", resampled_df)

    return resampled_df

In [10]:
start_time = datetime.strptime("202503251031", "%Y%m%d%H%M")
file_name = "test_speed"
file_path = "../assets/speed_tests/SCU1VP_202503251031_001.mp4"
track_history = defaultdict(lambda: [])
crossed_objects = {"EB": {}}
frame_count = 0
speeds = defaultdict(lambda: [])
class_ids = {2: "car", 5: "bus", 7: "truck"}

config = {
    "class_ids": class_ids,
    "tracker_config": tracker_config,
    "conf": confidence_threshold,
    "iou": 0.5,
}

cap = cv2.VideoCapture(file_path)
assert cap.isOpened(), "Error reading video file"
w, h, fps = (
    int(cap.get(x))
    for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS)
)


logging.info("frame_width: " + str(w))
logging.info("frame_height: " + str(h))

frame_generator = sv.get_video_frames_generator(source_path=file_path)

# Open a video sink for the output video
video_info = sv.VideoInfo.from_video_path(file_path)
if not pathlib.Path("../video/").exists():
    pathlib.Path("../video/").mkdir(parents=True, exist_ok=True)

video_report_path = "../video/" + "test_1.mp4"
with sv.VideoSink(video_report_path, video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):
        success, frame = cap.read()
        frame_count += 1
        if success:
            # undistort the frame
            # frame = undistort(frame)
            annotated_frame = get_count(
                frame,
                start_time,
                config,
                speeds,
                model,
                speed_zone,
                crossed_objects,
                depth_map,
                track_history,
                frame_count,
            )

            # convert annotated frame to distored frame
            annotated_frame = np.array(annotated_frame)
            annotated_frame = cv2.resize(annotated_frame, (w, h))

            # Show the frame with annotations
            # cv2.imshow("Frame", annotated_frame)
            # cv2.waitKey(1)

            # Draw the line on the frame
            # cv2.polylines(
            #    annotated_frame, [np.array(speed_zone, np.int32)], True, (0, 255, 0), 2
            # )

            # Write the count of objects on each frame
            # count_text_1 = f"Objects crossed EB: {len(crossed_objects['EB'])}"
            # cv2.putText(
            #     annotated_frame,
            #     count_text_1,
            #     (10, 30),
            #     cv2.FONT_HERSHEY_SIMPLEX,
            #     1,
            #     (0, 255, 0),
            #     2,
            # )

            # Write the frame with annotations to the output video
            sink.write_frame(annotated_frame)

        else:
            pass

# Release the video capture
cap.release()
print(f"Data has been written to {video_report_path}")

  6%|▋         | 536/8432 [01:12<17:55,  7.34it/s]

Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (1110.5439453125, 406.3139953613281)
 undst_pt_1: [     700.67      51.728]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (1099.588134765625, 403.0091247558594)
 undst_pt_1: [     693.75      51.307]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)


  6%|▋         | 538/8432 [01:12<18:07,  7.26it/s]

Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (1090.6558837890625, 398.4847106933594)
 undst_pt_1: [     688.11       50.73]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (1076.443115234375, 394.5619201660156)
 undst_pt_1: [     679.14       50.23]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)


  6%|▋         | 540/8432 [01:13<18:10,  7.23it/s]

...

Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (232.51071166992188, 244.33102416992188)
 undst_pt_1: [      144.4       31.06]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)


 30%|███       | 2561/8432 [05:51<13:20,  7.33it/s]

Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (227.603759765625, 242.62310791015625)
 undst_pt_1: [     141.03      30.835]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)
 track[-1]: (224.56163024902344, 242.34304809570312)
 undst_pt_1: [     138.93      30.797]
Frame shape: (720, 1280, 3)
Point matrix shape: (2,)


 33%|███▎      | 2788/8432 [06:23<12:55,  7.28it/s]


KeyboardInterrupt: 